### Test for the whole workflow

## Setup
Set up file system for the datset(using Google Drive), dagshub and MLflow

In [1]:
# Install dependencies
%pip install -q dagshub[jupyter]
!pip install mlflow
!pip install rasterio
!pip install natsort

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.2/203.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.2/28.2 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.8/231.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114

In [2]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Set up dagshub
!dagshub login

import dagshub
TOKEN = dagshub.auth.get_token()

                                ❗❗❗ AUTHORIZATION REQUIRED ❗❗❗                                


Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=1e45a5ef-01b4-4b5d-bdf5-4694849101e5&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=afcb5d128b442656d508b7668e2fc08d2f0df88cd46c430a765cff9823b3bf1f


⠏ Waiting for authorization
✅ OAuth token added


Accessing as chengzwk

In [4]:
# Suppress warnings
import logging
logging.getLogger("rasterio").setLevel(logging.ERROR)

In [9]:
# Read data
import os
import rasterio
import numpy as np
from natsort import natsorted

data_dir = "/content/drive/MyDrive/Omdena/urban-green-frankfurt/MULC"
image_dir = 'VBWVA_8R'
image_files = [f for f in os.listdir(os.path.join(data_dir, image_dir)) if f.endswith('1_GeoTIFF.tif')]
image_files = natsorted(image_files)
image_path = os.path.join(data_dir, image_dir, image_files[0])
with (rasterio.open(image_path) as img):
    image_array = img.read()
image_array = np.transpose(image_array, [1, 2, 0])  # move the axis for bands to the third axis
image_array[np.isnan(image_array)] = 0              # replace nan with 0
image_array = image_array[:, :, (1, 2, 3)]

# Create a directory for the results in Google Drive
results_dir = '/content/drive/My Drive/dummy_results'
os.makedirs(results_dir, exist_ok=True)

# Make a plot
import matplotlib.pyplot as plt
plt.imshow(image_array[:, :, 1])
plt.savefig(os.path.join(results_dir, 'image.png'))
plt.close()

# Create a dummy numpy array of ~1MB and save it to the results directory
np.save(os.path.join(results_dir, 'image_array.npy'), image_array)

print(f"results saved to: {results_dir}")

results saved to: /content/drive/My Drive/dummy_results


In [ ]:
from dagshub.notebook import save_notebook

repo = "chengzwk/omdena-frankfurt-ugs-unet"
branch = "overfit-experiments"
notebook_path = "unet-from-scratch.ipynb"
commit_message = "Test whole workflow"

save_notebook(
    repo=repo,
    branch=branch,
    path=notebook_path,
    commit_message=commit_message,
    versioning="git"
    )